# 05 Batch Correction and PCA

This notebook visualizes dataset-level batch effects before and after ComBat correction using PCA. It does not train classification models.

> **ComBat note:** ComBat is used to reduce dataset-level batch effects. In `src/apply_combat_correction.R`, the model matrix includes the asthma/control label with `model.matrix(~ label)`, which is intended to preserve biological label information while adjusting for dataset/batch effects.

## 1. Setup

Import packages, define input/output paths, and create output directories.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
processed_dir = project_root / "data" / "processed"
figures_dir = project_root / "figures"
results_dir = project_root / "results"

uncorrected_path = processed_dir / "merged_expression_common_genes.csv"
corrected_path = processed_dir / "merged_expression_combat_corrected.csv"
labels_path = processed_dir / "merged_labels.csv"
batch_labels_path = processed_dir / "merged_batch_labels.csv"

pca_before_batch_path = figures_dir / "pca_before_combat_by_batch.png"
pca_after_batch_path = figures_dir / "pca_after_combat_by_batch.png"
pca_before_label_path = figures_dir / "pca_before_combat_by_label.png"
pca_after_label_path = figures_dir / "pca_after_combat_by_label.png"
summary_path = results_dir / "pca_combat_summary.csv"

figures_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

## 2. Load Inputs

Load the uncorrected merged expression matrix, ComBat-corrected expression matrix, asthma/control labels, and dataset batch labels.

In [ ]:
for path in [uncorrected_path, corrected_path, labels_path, batch_labels_path]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

uncorrected = pd.read_csv(uncorrected_path, index_col=0)
corrected = pd.read_csv(corrected_path, index_col=0)
labels = pd.read_csv(labels_path, index_col=0)
batch_labels = pd.read_csv(batch_labels_path, index_col=0)

uncorrected = uncorrected.apply(pd.to_numeric, errors="coerce")
corrected = corrected.apply(pd.to_numeric, errors="coerce")

print(f"Uncorrected expression shape: {uncorrected.shape}")
print(f"ComBat-corrected expression shape: {corrected.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Batch labels shape: {batch_labels.shape}")

## 3. Align by Sample ID

Align all tables using the shared sample IDs so PCA coordinates, labels, and batch labels refer to the same samples.

In [ ]:
required_label_cols = {"label"}
required_batch_cols = {"dataset"}

if not required_label_cols.issubset(labels.columns):
    raise ValueError("merged_labels.csv must contain a 'label' column.")
if not required_batch_cols.issubset(batch_labels.columns):
    raise ValueError("merged_batch_labels.csv must contain a 'dataset' column.")

common_samples = uncorrected.index.intersection(corrected.index).intersection(labels.index).intersection(batch_labels.index)
if common_samples.empty:
    raise ValueError("No shared sample IDs found across all input files.")

uncorrected = uncorrected.loc[common_samples].sort_index()
corrected = corrected.loc[common_samples].sort_index()
labels = labels.loc[common_samples].sort_index()
batch_labels = batch_labels.loc[common_samples].sort_index()

if list(uncorrected.index) != list(corrected.index):
    raise ValueError("Uncorrected and corrected expression matrices are not aligned.")

if not uncorrected.columns.equals(corrected.columns):
    raise ValueError("Uncorrected and corrected matrices do not have the same gene columns.")

if uncorrected.isna().any().any() or corrected.isna().any().any():
    raise ValueError("Expression matrices contain missing or non-numeric values.")

print(f"Aligned samples: {len(common_samples)}")
print(f"Aligned genes: {uncorrected.shape[1]}")
print("\nBatch counts:")
print(batch_labels["dataset"].value_counts())
print("\nLabel counts:")
print(labels["label"].value_counts())

## 4. Standardize and Run PCA

Standardize each matrix and fit two-component PCA models before and after ComBat correction.

In [ ]:
def run_pca(expression_df, label):
    scaler = StandardScaler()
    scaled = scaler.fit_transform(expression_df)

    pca = PCA(n_components=2, random_state=3888)
    coords = pca.fit_transform(scaled)

    pca_df = pd.DataFrame(coords, index=expression_df.index, columns=["PC1", "PC2"])
    pca_df.index.name = "sample_id"
    pca_df["label"] = labels.loc[pca_df.index, "label"]
    pca_df["dataset"] = batch_labels.loc[pca_df.index, "dataset"]
    pca_df["matrix"] = label

    return pca, pca_df


pca_before, pca_before_df = run_pca(uncorrected, "Before ComBat")
pca_after, pca_after_df = run_pca(corrected, "After ComBat")

before_var = pca_before.explained_variance_ratio_
after_var = pca_after.explained_variance_ratio_

print(f"Before ComBat PC1 explained variance ratio: {before_var[0]:.4f}")
print(f"Before ComBat PC2 explained variance ratio: {before_var[1]:.4f}")
print(f"After ComBat PC1 explained variance ratio: {after_var[0]:.4f}")
print(f"After ComBat PC2 explained variance ratio: {after_var[1]:.4f}")

## 5. Plot PCA by Batch

Visualize whether samples cluster by dataset before and after ComBat correction.

In [ ]:
def plot_pca(pca_df, color_col, title, output_path, variance_ratio, color_map=None):
    fig, ax = plt.subplots(figsize=(7, 5))

    for group_name, group in pca_df.groupby(color_col):
        ax.scatter(
            group["PC1"],
            group["PC2"],
            label=group_name,
            color=None if color_map is None else color_map.get(group_name),
            alpha=0.8,
            edgecolor="white",
            linewidth=0.4,
        )

    ax.set_title(title)
    ax.set_xlabel(f"PC1 ({variance_ratio[0] * 100:.1f}% variance)")
    ax.set_ylabel(f"PC2 ({variance_ratio[1] * 100:.1f}% variance)")
    ax.legend(title=color_col)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

    fig.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved figure to {output_path}")


batch_colors = {
    "GSE40732": "#4C78A8",
    "GSE40888": "#F58518",
    "GSE64913": "#54A24B",
}

plot_pca(
    pca_before_df,
    color_col="dataset",
    title="PCA Before ComBat by Dataset",
    output_path=pca_before_batch_path,
    variance_ratio=before_var,
    color_map=batch_colors,
)

plot_pca(
    pca_after_df,
    color_col="dataset",
    title="PCA After ComBat by Dataset",
    output_path=pca_after_batch_path,
    variance_ratio=after_var,
    color_map=batch_colors,
)

## 6. Plot PCA by Asthma/Control Label

Visualize asthma/control structure before and after ComBat correction.

In [ ]:
label_colors = {
    "Control": "#4C78A8",
    "Asthma": "#F58518",
}

plot_pca(
    pca_before_df,
    color_col="label",
    title="PCA Before ComBat by Label",
    output_path=pca_before_label_path,
    variance_ratio=before_var,
    color_map=label_colors,
)

plot_pca(
    pca_after_df,
    color_col="label",
    title="PCA After ComBat by Label",
    output_path=pca_after_label_path,
    variance_ratio=after_var,
    color_map=label_colors,
)

## 7. Save PCA Summary

Save explained variance ratios and sample/gene counts for reporting.

In [ ]:
summary_df = pd.DataFrame(
    [
        {
            "matrix": "Before ComBat",
            "samples": uncorrected.shape[0],
            "genes": uncorrected.shape[1],
            "pc1_explained_variance_ratio": before_var[0],
            "pc2_explained_variance_ratio": before_var[1],
        },
        {
            "matrix": "After ComBat",
            "samples": corrected.shape[0],
            "genes": corrected.shape[1],
            "pc1_explained_variance_ratio": after_var[0],
            "pc2_explained_variance_ratio": after_var[1],
        },
    ]
)

summary_df.to_csv(summary_path, index=False)
print(f"Saved PCA summary to {summary_path}")
display(summary_df)